In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn import tree
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten,Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix




## Load data

In [2]:
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]

## Tree

In [3]:
tree_classifier=tree.DecisionTreeClassifier()
scores = cross_validate(tree_classifier, train_data_X, train_data_y, cv=5)

In [ ]:
print(scores)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([ 9.92942882, 10.07927227, 10.60560369, 11.47036314, 10.76992965]),
 'score_time': array([0.01673245, 0.01100111, 0.00800061, 0.00798082, 0.00700355]),
 'test_score': array([0.7775, 0.7815, 0.7805, 0.77  , 0.7715])}

In [6]:
print(scores)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([10.41250777, 10.65960598, 10.82245636, 11.84291673, 10.91298938]), 'score_time': array([0.01871014, 0.00700068, 0.0080061 , 0.00700402, 0.00799608]), 'test_score': array([0.7765, 0.789 , 0.783 , 0.777 , 0.7945])}
test_score_avg -  0.784


## FFN

In [8]:
def create_model(input_dim):
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Output layer for binary classification
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def create_multiclass_model(input_dim, num_classes):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))  # First hidden layer
    model.add(Dense(64, activation='relu'))  # Second hidden layer
    model.add(Dense(32, activation='relu'))  # Third hidden layer
    model.add(Dense(num_classes, activation='softmax'))  # Output layer for multi-class classification
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
                  metrics=['accuracy'])
    return model

In [14]:

# Set up 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index], train_data_X[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 1 Accuracy: 0.8035
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.82      0.71      0.76       415
           1       0.98      0.96      0.97       379
           2       0.78      0.89      0.83       388
           3       0.96      0.73      0.83       405
           4       0.59      0.75      0.66       413

    accuracy                           0.80      2000
   macro avg       0.83      0.81      0.81      2000
weighted avg       0.82      0.80      0.81      2000

Confusion Matrix for Fold 1:
 [[294   1  20   2  98]
 [  1 364   5   4   5]
 [  2   1 346   2  37]
 [ 21   4  13 295  72]
 [ 41   1  60   3 308]]
Training fold 2


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 2 Accuracy: 0.8325
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.79      0.79      0.79       398
           1       0.97      0.98      0.98       393
           2       0.91      0.78      0.84       425
           3       0.87      0.91      0.89       396
           4       0.65      0.70      0.67       388

    accuracy                           0.83      2000
   macro avg       0.84      0.83      0.83      2000
weighted avg       0.84      0.83      0.83      2000

Confusion Matrix for Fold 2:
 [[316   3   1  28  50]
 [  0 387   0   5   1]
 [  4   2 331   6  82]
 [ 14   7   2 359  14]
 [ 68   1  30  17 272]]
Training fold 3


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 3 Accuracy: 0.8265
Classification Report for Fold 3:
               precision    recall  f1-score   support

           0       0.82      0.71      0.76       399
           1       0.97      0.97      0.97       398
           2       0.76      0.90      0.82       366
           3       0.88      0.90      0.89       398
           4       0.71      0.68      0.70       439

    accuracy                           0.83      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.83      0.83      0.82      2000

Confusion Matrix for Fold 3:
 [[282   6  18  20  73]
 [  2 387   1   6   2]
 [  3   0 328   4  31]
 [ 14   5   8 357  14]
 [ 41   0  78  21 299]]
Training fold 4


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 4 Accuracy: 0.8285
Classification Report for Fold 4:
               precision    recall  f1-score   support

           0       0.81      0.82      0.81       411
           1       0.97      0.93      0.95       414
           2       0.77      0.88      0.82       406
           3       0.90      0.88      0.89       396
           4       0.69      0.61      0.65       373

    accuracy                           0.83      2000
   macro avg       0.83      0.82      0.82      2000
weighted avg       0.83      0.83      0.83      2000

Confusion Matrix for Fold 4:
 [[335   2  14  15  45]
 [  6 386   4  10   8]
 [  3   0 357   6  40]
 [ 18   8   8 350  12]
 [ 54   0  80  10 229]]
Training fold 5


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 5 Accuracy: 0.8410
Classification Report for Fold 5:
               precision    recall  f1-score   support

           0       0.85      0.77      0.81       410
           1       0.97      0.94      0.96       363
           2       0.83      0.87      0.85       416
           3       0.88      0.89      0.89       410
           4       0.69      0.74      0.72       401

    accuracy                           0.84      2000
   macro avg       0.85      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000

Confusion Matrix for Fold 5:
 [[316   1   7  14  72]
 [  4 342   4  12   1]
 [  7   1 361   7  40]
 [ 13   7   5 366  19]
 [ 32   0  56  16 297]]

Average Accuracy across 5 folds: 0.8264


In [ ]:
# no
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index], train_data_X[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=5)
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = (model.predict(X_val) > 0.5).astype("int32")
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)
    
    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

In [ ]:
fnn_classifier = create_multiclass_model()

In [ ]:
scores

## CNN

In [18]:

def create_cnn_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # Softmax for multi-class classification
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index].reshape(-1, 28, 28,1), train_data_X[val_index].reshape(-1, 28, 28,1)
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    print(X_train[0])
    
    # Create a new instance of the CNN model
    model = create_cnn_model(input_shape=train_data_X.reshape(-1, 28, 28, 1).shape[1:], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Convert probabilities to class predictions
    
    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1
[[[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [ 57]
  [176]
  [154]
  [150]
  [159]
  [ 92]
  [ 88]
  [ 48]
  [ 60]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [124]
  [173]
  [153]
  [133]
  [201]
  [114]
  [111]
  [ 87]
  [ 76]
  [ 19]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [159]
  [153]
  [102]
  [ 63]
  [119]
  [ 74]
  [ 76]
  [ 93]
  [ 81]
  [ 20]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [210]
  [114]
  [107]
  [119]
  [102]
  [ 70]
  [101]
  [ 59]
  [ 88]
  [ 51]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [ 32]


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Fold 1 Accuracy: 0.8595
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.85      0.79      0.82       415
           1       0.99      0.96      0.97       379
           2       0.90      0.86      0.88       388
           3       0.95      0.90      0.92       405
           4       0.68      0.80      0.73       413

    accuracy                           0.86      2000
   macro avg       0.87      0.86      0.86      2000
weighted avg       0.87      0.86      0.86      2000

Confusion Matrix for Fold 1:
 [[328   0   7   3  77]
 [  1 363   3   6   6]
 [  3   0 332   1  52]
 [ 11   4   2 365  23]
 [ 45   1  26  10 331]]
Training fold 2
[[[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  3]
  [  0]
  [  0]
  [  0]
  [ 41]
  [ 92]
  [172]
  [100]
  [ 23]
  [ 22]
  [ 33]
  [104]
  [136]
  [ 93]
  [ 44]
  [  0]
  [  0]
  [  0]
  [  2]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
 

c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Fold 2 Accuracy: 0.8530
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.83      0.72      0.77       398
           1       0.98      0.99      0.98       393
           2       0.91      0.83      0.87       425
           3       0.92      0.92      0.92       396
           4       0.66      0.80      0.72       388

    accuracy                           0.85      2000
   macro avg       0.86      0.85      0.85      2000
weighted avg       0.86      0.85      0.85      2000

Confusion Matrix for Fold 2:
 [[288   0   8  13  89]
 [  0 388   1   3   1]
 [ 12   0 353   5  55]
 [  6   6   3 365  16]
 [ 40   2  23  11 312]]
Training fold 3
[[[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  3]
  [  0]
  [  0]
  [  0]
  [ 41]
  [ 92]
  [172]
  [100]
  [ 23]
  [ 22]
  [ 33]
  [104]
  [136]
  [ 93]
  [ 44]
  [  0]
  [  0]
  [  0]
  [  2]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
 

...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Fold 3 Accuracy: 0.8585
Classification Report for Fold 3:
               precision    recall  f1-score   support

           0       0.82      0.81      0.82       399
           1       0.99      0.96      0.98       398
           2       0.87      0.84      0.85       366
           3       0.86      0.96      0.91       398
           4       0.76      0.73      0.75       439

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000

Confusion Matrix for Fold 3:
 [[325   1  12  22  39]
 [  1 382   2  10   3]
 [  1   0 306   8  51]
 [  3   1   2 382  10]
 [ 64   1  30  22 322]]
Training fold 4
[[[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  3]
  [  0]
  [  0]
  [  0]
  [ 41]
  [ 92]
  [172]
  [100]
  [ 23]
  [ 22]
  [ 33]
  [104]
  [136]
  [ 93]
  [ 44]
  [  0]
  [  0]
  [  0]
  [  2]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
 

c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Fold 4 Accuracy: 0.8540
Classification Report for Fold 4:
               precision    recall  f1-score   support

           0       0.84      0.85      0.84       411
           1       0.99      0.97      0.98       414
           2       0.82      0.87      0.84       406
           3       0.92      0.94      0.93       396
           4       0.68      0.62      0.65       373

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.85      0.85      0.85      2000

Confusion Matrix for Fold 4:
 [[350   1   4   7  49]
 [  0 402   0   5   7]
 [  3   1 352   5  45]
 [  9   0   5 371  11]
 [ 56   1  69  14 233]]
Training fold 5
[[[  0]
  [  0]
  [  0]
  [  0]
  [  0]
  [  3]
  [  0]
  [  0]
  [  0]
  [ 41]
  [ 92]
  [172]
  [100]
  [ 23]
  [ 22]
  [ 33]
  [104]
  [136]
  [ 93]
  [ 44]
  [  0]
  [  0]
  [  0]
  [  2]
  [  0]
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
 

c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Fold 5 Accuracy: 0.8530
Classification Report for Fold 5:
               precision    recall  f1-score   support

           0       0.83      0.83      0.83       410
           1       0.97      0.98      0.97       363
           2       0.85      0.83      0.84       416
           3       0.90      0.89      0.89       410
           4       0.73      0.75      0.74       401

    accuracy                           0.85      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.85      0.85      0.85      2000

Confusion Matrix for Fold 5:
 [[340   0   6  15  49]
 [  2 354   0   6   1]
 [ 16   2 347   3  48]
 [ 11   7  12 364  16]
 [ 39   2  42  17 301]]

Average Accuracy across 5 folds: 0.8556
